# Heart sound classification (log-mel CNN)
Author: Abdullah Ghani, ECE Paris (Data & AI)

Binary classification of short phonocardiogram recordings: normal vs abnormal (murmur, extra heart sound, extrasystole). Public dataset: Kaggle `kinguistics/heartbeat-sounds` (PASCAL Classifying Heart Sounds Challenge, sets A and B). Noisy "artifact" recordings are excluded from the task.
Because the dataset is small, results use **5-fold stratified cross-validation** (out-of-fold predictions, bootstrap 95% CI).

> Research prototype. Not a medical device. Not for clinical use. Runtime: Colab (GPU optional).

In [ ]:
!pip install -q kagglehub librosa
import os, glob, json, random, numpy as np, pandas as pd, tensorflow as tf, librosa, matplotlib.pyplot as plt
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
SEED=42; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
import kagglehub
root=kagglehub.dataset_download("kinguistics/heartbeat-sounds")
for r,d,f in os.walk(root): print(r, d[:4], f[:3])

In [ ]:
wavs=glob.glob(root+"/**/*.wav",recursive=True); print(len(wavs))
def lab_of(p):
    n=os.path.basename(p).lower(); par=os.path.basename(os.path.dirname(p)).lower()
    for k in ["normal","murmur","extrahls","artifact","extrastole"]:
        if n.startswith(k) or k in par: return k
    return None
rows=[(p,lab_of(p)) for p in wavs]; df=pd.DataFrame(rows,columns=["path","label"]).dropna()
print(df.label.value_counts())
df=df[df.label!="artifact"].reset_index(drop=True)
df["y"]=(df.label!="normal").astype(int); print(df.y.value_counts())

In [ ]:
SR=2000; DUR=5; N_MELS=64
def feat(p):
    y,_=librosa.load(p,sr=SR,mono=True,duration=DUR*2)
    y=y[:SR*DUR]; y=np.pad(y,(0,max(0,SR*DUR-len(y))))
    y=y/(np.abs(y).max()+1e-8)
    m=librosa.power_to_db(librosa.feature.melspectrogram(y=y,sr=SR,n_fft=256,hop_length=32,n_mels=N_MELS,fmax=SR//2),ref=np.max)
    return ((m+80)/80).astype("float32")
X=np.stack([feat(p) for p in df.path])[...,None]; y=df.y.values; print(X.shape, np.bincount(y))

In [ ]:
def build():
    inp=layers.Input(X.shape[1:])
    x=layers.RandomTranslation(0,0.1)(inp)
    for f in [16,32,64]:
        x=layers.Conv2D(f,3,padding="same",activation="relu")(x); x=layers.BatchNormalization()(x); x=layers.MaxPool2D(2)(x)
    x=layers.GlobalAveragePooling2D()(x); x=layers.Dropout(0.4)(x); out=layers.Dense(1,activation="sigmoid")(x)
    m=models.Model(inp,out); m.compile(optimizers.Adam(1e-3),"binary_crossentropy"); return m
cw={0:len(y)/(2*(y==0).sum()),1:len(y)/(2*(y==1).sum())}
skf=StratifiedKFold(5,shuffle=True,random_state=SEED); oof=np.zeros(len(y))
for k,(tr,va) in enumerate(skf.split(X,y)):
    tf.keras.backend.clear_session(); m=build(); m.fit(X[tr],y[tr],epochs=40,batch_size=16,class_weight=cw,verbose=0)
    oof[va]=m.predict(X[va],verbose=0).ravel()
    print("fold",k,"AUC",round(roc_auc_score(y[va],oof[va]),3))

In [ ]:
pred=(oof>0.5).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred).ravel()
def stats(yt,pp,ps):
    tn,fp,fn,tp=confusion_matrix(yt,pp,labels=[0,1]).ravel()
    return dict(acc=(tp+tn)/len(yt),sens=tp/max(tp+fn,1),spec=tn/max(tn+fp,1),auc=roc_auc_score(yt,ps))
pt=stats(y,pred,oof); rng=np.random.default_rng(0); bs=[]
for _ in range(1000):
    i=rng.integers(0,len(y),len(y))
    if len(set(y[i]))<2: continue
    bs.append(stats(y[i],pred[i],oof[i]))
ci={k:[float(np.percentile([b[k] for b in bs],2.5)),float(np.percentile([b[k] for b in bs],97.5))] for k in pt}
res={"n_recordings":int(len(y)),"protocol":"5-fold stratified CV, out-of-fold predictions","classes":"normal vs abnormal (murmur, extrahls, extrastole); artifact excluded","point":{k:float(v) for k,v in pt.items()},"ci95_bootstrap":ci,"confusion_matrix":{"TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp)}}
json.dump(res,open("metrics.json","w"),indent=2); print(json.dumps(res,indent=2))
cm=confusion_matrix(y,pred); fig,ax=plt.subplots(figsize=(4,4)); ax.imshow(cm,cmap="Blues")
for i in range(2):
    for j in range(2): ax.text(j,i,cm[i,j],ha="center",va="center")
ax.set_xticks([0,1]);ax.set_yticks([0,1]);ax.set_xticklabels(["normal","abnormal"]);ax.set_yticklabels(["normal","abnormal"]);ax.set_xlabel("predicted");ax.set_ylabel("true");ax.set_title("Out-of-fold confusion matrix")
plt.tight_layout(); plt.savefig("confusion_matrix.png",dpi=150); plt.show()
f,t,_=roc_curve(y,oof); plt.figure(figsize=(4,4)); plt.plot(f,t); plt.plot([0,1],[0,1],"--"); plt.xlabel("1 - specificity"); plt.ylabel("sensitivity"); plt.title(f"ROC (out-of-fold) AUC {pt['auc']:.3f}"); plt.tight_layout(); plt.savefig("roc_curve.png",dpi=150); plt.show()
i=int(np.where(y==1)[0][0]); j=int(np.where(y==0)[0][0])
fig,ax=plt.subplots(1,2,figsize=(9,3)); ax[0].imshow(X[j,...,0],origin="lower",aspect="auto"); ax[0].set_title("normal (log-mel)"); ax[1].imshow(X[i,...,0],origin="lower",aspect="auto"); ax[1].set_title("abnormal (log-mel)"); plt.tight_layout(); plt.savefig("spectrograms.png",dpi=130); plt.show()

In [ ]:
tf.keras.backend.clear_session(); m=build(); m.fit(X,y,epochs=40,batch_size=16,class_weight=cw,verbose=0); m.save("heart_sound_cnn.keras")
import zipfile
with zipfile.ZipFile("heart_results.zip","w") as z:
    for f in ["heart_sound_cnn.keras","metrics.json","confusion_matrix.png","roc_curve.png","spectrograms.png"]: z.write(f)
print("Saved heart_results.zip")

## Limitations
- A few hundred recordings; confidence intervals are wide.
- No patient identifiers, so recordings of one patient may fall in different folds and cross-validation can be optimistic.
- Short recordings from stethoscope apps and clinical devices with different noise; no external validation.
- Normal vs abnormal only: no murmur grading, no diagnosis.
- The saved demo model is trained on all recordings; the reported metrics come from the cross-validation.